In [1]:
import os

# Clear Kaggle TPU environment variables to prevent initialization errors
os.environ.pop('TPU_PROCESS_ADDRESSES', None)
os.environ.pop('CLOUD_TPU_TASK_ID', None)

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import timm
import shutil

SCALES_TO_RUN = ["10%", "25%", "50%", "100%"]
EPOCHS        = 200     # Maximum cap; early stopping prevents overfitting on small scales
PATIENCE      = 20      # Halt if no validation accuracy improvement after 20 epochs
LR            = 5e-4
WEIGHT_DECAY  = 0.05

TEMPERATURE   = 4.0
W_TASK        = 0.6     # Cross-Entropy hard label loss weight
W_DISTILL     = 0.4     # KL Divergence soft label loss weight
W_SPATIAL     = 0.5     # Attention / Grad-CAM MSE alignment loss weight

# Options: "live" (GPU only, recomputes Grad-CAM via autograd) 
#          "cached" (TPU/GPU, precomputes maps to disk using Cell 6)
GRADCAM_MODE  = "live"
CACHE_PATH    = "./gradcam_cache.npy"

PROJ_DIM_IN   = 192
PROJ_DIM_OUT  = 2048    # ResNet-50 features dimension

In [2]:
DEVICE_TYPE = "GPU"   # "GPU" | "TPU"

class DeviceManager:
    """Unified stub handling both multi-GPU DataParallel and TPU environments seamlessly."""
    def __init__(self):
        self.type = DEVICE_TYPE
        if self.type == "TPU":
            import torch_xla
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr
            import torch_xla.distributed.parallel_loader as pl
            self.xm = xm
            self.xr = xr
            self.pl = pl
            self.device = torch_xla.device()
            self.world_size = xr.world_size()
            self.rank = xr.global_ordinal()
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.world_size = 1
            self.rank = 0

    def step(self, opt):
        if self.type == "TPU":
            self.xm.optimizer_step(opt)
            self.xm.mark_step()
        else:
            opt.step()

    def wrap_loader(self, loader):
        if self.type == "TPU":
            return self.pl.MpDeviceLoader(loader, self.device)
        return loader

    def is_master(self):
        return self.rank == 0

    def master_print(self, *args, **kwargs):
        if self.is_master():
            print(*args, **kwargs)

In [3]:
BACKUP_PATH = "/kaggle/input/datasets/camphor5/checkpoints-failed-horse3-res/outputs" 
TARGET_DIR = "./outputs"

if os.path.exists(BACKUP_PATH):
    print("Restoring backup...")
    shutil.copytree(BACKUP_PATH, TARGET_DIR, dirs_exist_ok=True)
    print("Restore complete! Found files:")
    print("Checkpoints:", os.listdir(f"{TARGET_DIR}/checkpoints"))
    print("Results:", os.listdir(f"{TARGET_DIR}/results"))
else:
    print("Backup path not found. Check your dataset name.")

Restoring backup...
Restore complete! Found files:
Checkpoints: ['nb04_50pct.pth', 'nb04_10pct.pth', 'nb04_100pct.pth', 'nb04_25pct.pth']
Results: ['nb04_10pct.json', 'nb04_25pct.json', 'nb04_50pct.json']


In [4]:
TRAIN_DIR    = "/kaggle/input/datasets/melikechan/cifar100/cifar100/train"
TEST_DIR     = "/kaggle/input/datasets/melikechan/cifar100/cifar100/test"
TEACHER_CKPT = "/kaggle/input/datasets/totallyapoorv/resnet-teachermodel-50/teacher_resnet50.pth"

WORK_DIR = "./outputs"
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",     exist_ok=True)

In [5]:
SEED       = 67
BATCH_SIZE = 64
scale_map  = {"10%": 0.10, "25%": 0.25, "50%": 0.50, "100%": 1.0}
key_of     = lambda s: s.replace("%", "pct")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

def build_loaders(scale, ctx):
    tfm_tr = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    tfm_te = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    
    full = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=tfm_tr)
    test = torchvision.datasets.ImageFolder(TEST_DIR,  transform=tfm_te)
    
    idx = list(range(len(full)))
    random.Random(SEED).shuffle(idx)
    sub = torch.utils.data.Subset(full, idx[:int(len(full) * scale_map[scale])])
    
    sampler = torch.utils.data.distributed.DistributedSampler(
        sub, num_replicas=ctx.world_size, rank=ctx.rank, shuffle=True
    ) if ctx.world_size > 1 else None

    tr = torch.utils.data.DataLoader(
        sub, BATCH_SIZE, shuffle=(sampler is None), 
        sampler=sampler, num_workers=2, pin_memory=True
    )
    te = torch.utils.data.DataLoader(test, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, te, len(sub)

In [6]:
def evaluate(model, loader, ctx):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in ctx.wrap_loader(loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            out = model(x)
            
            # Handle 3-element tuples (cls_logits, dist_logits, attention_map)
            if isinstance(out, tuple) and len(out) == 3: 
                out = (out[0] + out[1]) / 2
            # Handle standard 2-element tuples
            elif isinstance(out, tuple):
                out = (out[0] + out[1]) / 2
            elif hasattr(out, 'cls_logits') and hasattr(out, 'dist_logits'):
                out = (out.cls_logits + out.dist_logits) / 2
                
            correct += out.argmax(1).eq(y).sum().item()
            total   += y.size(0)
    
    if ctx.type == "TPU":
        correct = ctx.xm.mesh_reduce("test_correct", correct, sum)
        total = ctx.xm.mesh_reduce("test_total", total, sum)
        
    return 100. * correct / total

def save_ckpt(path, epoch, model, proj, opt, best_acc, history, scheduler=None, extras=None):
    d = dict(epoch=epoch, model=model.state_dict(), opt=opt.state_dict(),
             best_acc=best_acc, history=history)
    if proj is not None: 
        d["proj"] = proj.state_dict()
    if scheduler is not None: 
        d["scheduler"] = scheduler.state_dict()
    if extras:           
        d.update(extras)
    torch.save(d, path)

def load_ckpt(path, model, proj, opt, ctx, scheduler=None):
    d  = torch.load(path, map_location='cpu')
    
    def strip_wrappers(state):
        cleaned = {}
        for k, v in state.items():
            new_key = k
            if new_key.startswith("ddp_module.module."): new_key = new_key[18:]
            if new_key.startswith("module."): new_key = new_key[7:]
            cleaned[new_key] = v
        return cleaned
        
    base_model = model.module if hasattr(model, 'module') else model
    base_model.load_state_dict(strip_wrappers(d["model"]))
    
    if proj is not None and "proj" in d: 
        base_proj = proj.module if hasattr(proj, 'module') else proj
        base_proj.load_state_dict(strip_wrappers(d["proj"]))
        
    opt.load_state_dict(d["opt"])
    
    if scheduler is not None and "scheduler" in d:
        scheduler.load_state_dict(d["scheduler"])
        
    return d["epoch"]+1, d["best_acc"], d.get("history", [])

In [7]:
class WrappedTeacher(nn.Module):
    """Calculates Grad-CAM without leaking memory graphs. Removed DataParallel overhead."""
    def __init__(self, base_teacher):
        super().__init__()
        self.base_teacher = base_teacher
        
    def forward(self, x, target_class=None):
        with torch.no_grad():
            x = self.base_teacher.conv1(x)
            x = self.base_teacher.bn1(x)
            x = self.base_teacher.relu(x)
            x = self.base_teacher.maxpool(x)
            x = self.base_teacher.layer1(x)
            x = self.base_teacher.layer2(x)
            x = self.base_teacher.layer3(x)
            feature_map = self.base_teacher.layer4(x)  # [B, 2048, 7, 7]
        
        if target_class is not None:
            with torch.enable_grad():
                feature_map = feature_map.clone().requires_grad_(True)
                pool_feat = self.base_teacher.avgpool(feature_map)
                pool_feat = torch.flatten(pool_feat, 1)
                logits = self.base_teacher.fc(pool_feat)
                
                score = logits.gather(1, target_class.view(-1, 1)).sum()
                grads = torch.autograd.grad(score, feature_map, retain_graph=False)[0]
                
                weights = grads.mean(dim=(2, 3), keepdim=True)
                cam = F.relu((weights * feature_map).sum(dim=1))  # [B, 7, 7]
                
                cam_flat = cam.view(cam.size(0), -1)
                cam_norm = F.normalize(cam_flat, p=2, dim=1).view_as(cam)
                
            return logits.detach(), cam_norm.detach()
        else:
            with torch.no_grad():
                pool_feat = self.base_teacher.avgpool(feature_map)
                pool_feat = torch.flatten(pool_feat, 1)
                logits = self.base_teacher.fc(pool_feat)
            return logits.detach(), None

class WrappedStudent(nn.Module):
    """Thread-safe wrapper intercepting distillation token attention maps across multi-GPUs."""
    def __init__(self, base_student):
        super().__init__()
        self.base_student = base_student
        self.base_student.blocks[-1].attn.fused_attn = False
        
    def forward(self, x):
        attn_weights = None
        
        def local_hook(module, inp, out):
            nonlocal attn_weights
            attn_weights = out[:, :, 1, 2:].mean(dim=1)
            
        handle = self.base_student.blocks[-1].attn.attn_drop.register_forward_hook(local_hook)
        features = self.base_student.forward_features(x)
        handle.remove()
        
        cls_token = features[:, 0]
        dist_token = features[:, 1]
        
        cls_logits = self.base_student.head(cls_token)
        dist_logits = self.base_student.head_dist(dist_token)
        return cls_logits, dist_logits, attn_weights

def build_teacher(ctx):
    t = torchvision.models.resnet50(weights=None)
    t.fc = nn.Linear(t.fc.in_features, 100)
    t.load_state_dict(torch.load(TEACHER_CKPT, map_location='cpu'))
    t = WrappedTeacher(t).to(ctx.device).eval()
    for p in t.parameters(): 
        p.requires_grad_(False)
    
    # CRITICAL FIX: The Teacher is explicitly NOT wrapped in DataParallel
    return t

def build_student(ctx):
    m = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=False, num_classes=100)
    m = WrappedStudent(m).to(ctx.device)
    
    if ctx.type == "GPU" and torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)

    if ctx.is_master():
        total_params = sum(p.numel() for p in m.parameters())
        print(f"  Student params: {total_params/1e6:.2f}M")
    return m

In [8]:
def spatial_loss(s_attn, teacher_cam):
    """Bilinearly interpolates, normalizes, and computes the MSE alignment loss between maps."""
    # Student attention: [B, 196] -> [B, 1, 14, 14]
    s = s_attn.view(-1, 1, 14, 14).float()
    # Teacher Grad-CAM: [B, 7, 7] -> [B, 1, 14, 14]
    t = F.interpolate(teacher_cam.unsqueeze(1), size=(14, 14), mode="bilinear", align_corners=False)
    
    s_n = F.normalize(s.view(s.size(0), -1), p=2, dim=1)
    t_n = F.normalize(t.view(t.size(0), -1), p=2, dim=1)
    return F.mse_loss(s_n, t_n)

In [9]:
class CachedDataset(torch.utils.data.Dataset):
    """Dataset wrapper connecting offline precomputed Grad-CAM arrays to image inputs."""
    def __init__(self, subset, cache_array):
        self.subset = subset
        self.cache_array = cache_array
    def __len__(self):  
        return len(self.subset)
    def __getitem__(self, i):
        img, lbl = self.subset[i]
        orig_idx = self.subset.indices[i]
        return img, lbl, self.cache_array[orig_idx]

if GRADCAM_MODE == "cached" and DeviceManager().is_master():
    print("Pre-computing teacher Grad-CAM maps for ALL training images...")
    tfm = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    full_ds = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=tfm)
    full_loader = torch.utils.data.DataLoader(full_ds, batch_size=64, shuffle=False, num_workers=2)
    
    _ctx = DeviceManager()
    _teacher = build_teacher(_ctx)
    all_cams = np.zeros((len(full_ds), 7, 7), dtype=np.float32)
    ptr = 0
    
    for x, y in full_loader:
        x, y = x.to(_ctx.device), y.to(_ctx.device)
        with torch.enable_grad():
            _, cam = _teacher(x, target_class=y)
        bs = x.size(0)
        all_cams[ptr:ptr+bs] = cam.cpu().numpy()
        ptr += bs
        if ptr % 5000 == 0: 
            print(f"  {ptr}/{len(full_ds)}")
            
    np.save(CACHE_PATH, all_cams)
    print(f"Saved Grad-CAM cache to: {CACHE_PATH}, shape: {all_cams.shape}")
else:
    print("Live execution mode selected or sub-process worker skipping cache routines.")

Live execution mode selected or sub-process worker skipping cache routines.


In [10]:
def train_one_scale(index, scale):
    ctx = DeviceManager()
    key  = key_of(scale)
    ckpt = f"{WORK_DIR}/checkpoints/nb04_{key}.pth"
    rp   = f"{WORK_DIR}/results/nb04_{key}.json"

    if os.path.exists(rp) and json.load(open(rp)).get("completed"):
        r = json.load(open(rp))
        ctx.master_print(f"[{scale}] already done — best {r['best_acc']:.2f}%")
        return r

    tr_loader, te_loader, n_train = build_loaders(scale, ctx)
    
    if GRADCAM_MODE == "cached":
        cache_data = np.load(CACHE_PATH)
        full_trainset = torchvision.datasets.ImageFolder(
            TRAIN_DIR, transform=transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
            ]))
        idx = list(range(len(full_trainset)))
        random.Random(SEED).shuffle(idx)
        sub_idx = idx[:int(len(full_trainset) * scale_map[scale])]
        sub_ds = torch.utils.data.Subset(full_trainset, sub_idx)
        cached_ds = CachedDataset(sub_ds, cache_data)
        
        sampler = torch.utils.data.distributed.DistributedSampler(
            cached_ds, num_replicas=ctx.world_size, rank=ctx.rank, shuffle=True
        ) if ctx.world_size > 1 else None
        
        tr_loader = torch.utils.data.DataLoader(
            cached_ds, BATCH_SIZE, shuffle=(sampler is None),
            sampler=sampler, num_workers=2, pin_memory=True
        )

    teacher = build_teacher(ctx)
    student = build_student(ctx)
    opt     = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    start_epoch, best_acc, history = 0, 0.0, []
    epochs_no_improve = 0
    
    if os.path.exists(ckpt):
        # Load the scheduler state explicitly
        start_epoch, best_acc, history = load_ckpt(ckpt, student, None, opt, ctx, scheduler=scheduler)
        ctx.master_print(f"[{scale}] resumed at epoch {start_epoch+1}")

    ctx.master_print(f"[{scale}] training on {n_train} images")
    for epoch in range(start_epoch, EPOCHS):
        if ctx.type == "TPU" and hasattr(tr_loader, 'sampler') and tr_loader.sampler is not None:
            tr_loader.sampler.set_epoch(epoch)
            
        student.train()
        t0 = time.time()
        
        if ctx.type == "TPU":
            sums = {"total": torch.tensor(0., device=ctx.device), "task": torch.tensor(0., device=ctx.device), 
                    "distill": torch.tensor(0., device=ctx.device), "spatial": torch.tensor(0., device=ctx.device)}
        else:
            sums = {"total": 0., "task": 0., "distill": 0., "spatial": 0.}

        for batch in ctx.wrap_loader(tr_loader):
            opt.zero_grad()
            
            if GRADCAM_MODE == "cached":
                x, y, teacher_cam = batch
                x, y = x.to(ctx.device), y.to(ctx.device)
                teacher_cam = teacher_cam.to(ctx.device)
                t_logits, _ = teacher(x)
            else:
                x, y = batch
                x, y = x.to(ctx.device), y.to(ctx.device)
                # WrappedTeacher handles all gradients and detaching internally without DataParallel overhead
                t_logits, teacher_cam = teacher(x, target_class=y)

            cls_l, dist_l, s_attn = student(x)
                
            tl = F.cross_entropy(cls_l, y)
            dl = F.kl_div(F.log_softmax(dist_l / TEMPERATURE, dim=1),
                          F.softmax(t_logits / TEMPERATURE, dim=1),
                          reduction="batchmean") * (TEMPERATURE ** 2)
            sl = spatial_loss(s_attn, teacher_cam)
            
            loss = (W_TASK * tl) + (W_DISTILL * dl) + (W_SPATIAL * sl)
            loss.backward()
            ctx.step(opt)
            
            if ctx.type == "TPU":
                sums["total"] += loss.detach()
                sums["task"] += tl.detach()
                sums["distill"] += dl.detach()
                sums["spatial"] += sl.detach()
            else:
                sums["total"] += loss.item()
                sums["task"] += tl.item()
                sums["distill"] += dl.item()
                sums["spatial"] += sl.item()
                
            # FORCE memory clearing to prevent CPU RAM buildup
            del x, y, teacher_cam, t_logits, cls_l, dist_l, s_attn, tl, dl, sl, loss

        epoch_time = time.time() - t0
        scheduler.step()
        
        nb_ = len(tr_loader)
        if ctx.type == "TPU":
            avg_total = ctx.xm.mesh_reduce("tot_reduce", sums["total"].item(), sum) / (nb_ * ctx.world_size)
            avg_task  = ctx.xm.mesh_reduce("tsk_reduce", sums["task"].item(), sum) / (nb_ * ctx.world_size)
            avg_dist  = ctx.xm.mesh_reduce("dst_reduce", sums["distill"].item(), sum) / (nb_ * ctx.world_size)
            avg_spat  = ctx.xm.mesh_reduce("spa_reduce", sums["spatial"].item(), sum) / (nb_ * ctx.world_size)
        elif ctx.type == "GPU" and ctx.world_size > 1:
            import torch.distributed as dist
            metrics = torch.tensor([sums["total"], sums["task"], sums["distill"], sums["spatial"]], dtype=torch.float32, device=ctx.device)
            dist.all_reduce(metrics, op=dist.ReduceOp.SUM)
            avg_total, avg_task, avg_dist, avg_spat = (metrics / (nb_ * ctx.world_size)).tolist()
        else:
            avg_total = sums["total"] / nb_
            avg_task  = sums["task"] / nb_
            avg_dist  = sums["distill"] / nb_
            avg_spat  = sums["spatial"] / nb_
            
        val_acc = evaluate(student, te_loader, ctx)
        
        if val_acc > best_acc:
            best_acc = val_acc
            epochs_no_improve = 0
            is_best = True
        else:
            epochs_no_improve += 1
            is_best = False

        if ctx.is_master():
            rec = {"epoch": epoch, "total": avg_total, "task": avg_task, "distill": avg_dist, 
                   "spatial": avg_spat, "val_acc": val_acc, "epoch_time": epoch_time}
            history.append(rec)
            ctx.master_print(f"[{scale}] E{epoch+1}/{EPOCHS} | total={avg_total:.4f} "
                             f"task={avg_task:.4f} kl={avg_dist:.4f} spatial={avg_spat:.4f} | "
                             f"val={val_acc:.2f}% | {epoch_time:.0f}s | wait={epochs_no_improve}/{PATIENCE}")
            if is_best:
                # Save the scheduler state explicitly
                save_ckpt(ckpt, epoch, student, None, opt, best_acc, history, scheduler=scheduler)

        if epochs_no_improve >= PATIENCE:
            ctx.master_print(f"[{scale}] Early stopping triggered at epoch {epoch+1}!")
            break

    if ctx.is_master():
        result = {"scale": scale, "n_train": n_train, "final_acc": history[-1]["val_acc"] if history else best_acc,
                  "best_acc": best_acc, "history": history, "completed": True}
        json.dump(result, open(rp, "w"), indent=2)
        ctx.master_print(f"[{scale}] DONE — final {result['final_acc']:.2f}% | best {best_acc:.2f}%")
        return result
    return None

In [11]:
def _mp_fn(index, scales):
    for scale in scales:
        train_one_scale(index, scale)

if __name__ == "__main__":
    if DEVICE_TYPE == "TPU":
        import torch_xla.distributed.xla_multiprocessing as xmp
        xmp.spawn(_mp_fn, args=(SCALES_TO_RUN,), start_method='fork')
    else:
        # Avoids multiprocessing complications on GPU infrastructure.
        # DataParallel natively scales over multi-GPU threads inside this single process.
        _mp_fn(0, SCALES_TO_RUN)
    print("All scales complete.")

[10%] already done — best 17.39%
[25%] already done — best 28.30%
[50%] already done — best 40.38%
  Student params: 5.56M
[100%] resumed at epoch 88
[100%] training on 50000 images
[100%] E88/200 | total=1.0315 task=0.1159 kl=2.4002 spatial=0.0037 | val=44.08% | 256s | wait=1/20
[100%] E89/200 | total=0.6063 task=0.0213 kl=1.4792 spatial=0.0038 | val=46.66% | 260s | wait=2/20
[100%] E90/200 | total=0.4457 task=0.0042 kl=1.1034 spatial=0.0037 | val=47.87% | 260s | wait=0/20
[100%] E91/200 | total=0.4717 task=0.0117 kl=1.1573 spatial=0.0035 | val=45.12% | 260s | wait=1/20
[100%] E92/200 | total=1.0119 task=0.1053 kl=2.3671 spatial=0.0038 | val=45.75% | 259s | wait=2/20
[100%] E93/200 | total=0.4814 task=0.0068 kl=1.1886 spatial=0.0037 | val=47.37% | 260s | wait=3/20
[100%] E94/200 | total=0.4280 task=0.0038 kl=1.0599 spatial=0.0036 | val=47.98% | 260s | wait=0/20
[100%] E95/200 | total=0.8013 task=0.0735 kl=1.8884 spatial=0.0036 | val=42.85% | 260s | wait=1/20
[100%] E96/200 | total=0.6

In [12]:
import os
import json

SCALES_TO_RUN = ["10%", "25%", "50%", "100%"]
WORK_DIR      = "./outputs"
key_of        = lambda s: s.replace("%", "pct")

print(f"\n{'Scale':<8} {'Final Acc':>10} {'Best Acc':>10} {'Train (s)':>12}")
for scale in SCALES_TO_RUN:
    rp = f"{WORK_DIR}/results/nb04_{key_of(scale)}.json"
    if os.path.exists(rp):
        try:
            with open(rp, "r") as f:
                r = json.load(f)
            wall = sum(h.get("epoch_time", 0) for h in r.get("history", []))
            print(f"{scale:<8} {r.get('final_acc', 0.0):>9.2f}% {r.get('best_acc', 0.0):>9.2f}% {wall:>11.0f}s")
        except Exception:
            print(f"{scale:<8} error reading results json")
    else:
        print(f"{scale:<8} not yet complete")


Scale     Final Acc   Best Acc    Train (s)
10%          16.68%     17.39%         921s
25%          27.18%     28.30%        3714s
50%          39.96%     40.38%       22501s
100%         49.11%     49.67%       44362s
